In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import uuid

posts = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset/anxiety_post.csv")
comments = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset/anxiety_comment.csv")

all_authors = pd.concat([posts["Post_Author"], comments["Comment_Author"]]).unique()
author_map = {author: str(uuid.uuid4()) for author in all_authors}

In [ ]:
print(author_map)

In [ ]:
posts_clean = posts[["Post_ID","Post_Content","Post_Author","Post_Date","Post_Time","Post_Category"]].copy()
posts_clean["Author_ID"] = posts_clean["Post_Author"].map(author_map)
posts_clean['Post_Date'] = posts_clean['Post_Date'].str.replace(u'\u200e', '', regex=True)
posts_clean['Post_Time'] = posts_clean['Post_Time'].str.replace(u'\u200e', '', regex=True)
posts_clean["Timestamp"] = pd.to_datetime(posts_clean["Post_Date"] + " " + posts_clean["Post_Time"], format='%d-%m-%Y %I:%M %p', errors="coerce")
posts_clean = posts_clean.drop(columns=["Post_Author","Post_Date","Post_Time", "Post_Category"])
posts_clean = posts_clean.dropna(subset=["Post_Content","Timestamp"])
posts_clean = posts_clean.drop_duplicates(subset=["Post_ID"])
print(posts_clean)

In [ ]:
comments_clean = comments[["Post_ID","Comment_ID","Comment_Content","Comment_Author","Comment_Date","Comment_Time"]].copy()
comments_clean["Author_ID"] = comments_clean["Comment_Author"].map(author_map).fillna("00000000-0000-0000-0000-000000000000")
comments_clean['Comment_Date'] = comments_clean['Comment_Date'].str.replace(u'\u200e', '', regex=True)
comments_clean['Comment_Time'] = comments_clean['Comment_Time'].str.replace(u'\u200e', '', regex=True)
comments_clean["Timestamp"] = pd.to_datetime(comments_clean["Comment_Date"] + " " + comments_clean["Comment_Time"], errors="coerce")
comments_clean = comments_clean.drop(columns=["Comment_Author","Comment_Date","Comment_Time"])
comments_clean = comments_clean.dropna(subset=["Comment_Content","Timestamp"])
comments_clean = comments_clean.drop_duplicates(subset=["Comment_ID"])
comments_clean = comments_clean[comments_clean["Post_ID"].isin(posts_clean["Post_ID"])]
comments_clean = comments_clean.sort_values(["Post_ID","Timestamp"])
print(comments_clean)

In [ ]:
import json
save_location = "/content/drive/MyDrive/Colab Notebooks/MastersResearchPrelim/dataset_clean/"
with open(save_location + "author_map_SECURE.json", "w") as f:
    json.dump(author_map, f)

posts_clean.to_csv(save_location + "posts_cleaned.csv", index=False)
comments_clean.to_csv(save_location + "comments_cleaned.csv", index=False)

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from wordcloud import WordCloud, STOPWORDS
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

# ── CONFIG ────────────────────────────────────────────────────────────────────

DATA_LOCATION = "/content/drive/MyDrive/Colab Notebooks/MastersResearchPrelim/dataset_clean/"

POSTS_CSV    = "posts_cleaned.csv"
COMMENTS_CSV = "comments_cleaned.csv"

# Column names — change these if your CSV uses different headers
POST_ID_COL       = "Post_ID"
POST_CONTENT_COL  = "Post_Content"
POST_TS_COL       = "Timestamp"       # parsed datetime column in posts

COMMENT_POST_ID_COL  = "Post_ID"
COMMENT_CONTENT_COL  = "Comment_Content"
COMMENT_TS_COL       = "Timestamp"    # parsed datetime column in comments

# Histogram bins
N_BINS = 30

# Word cloud dimensions
WC_WIDTH, WC_HEIGHT = 1600, 800

# Extra domain stopwords — add any forum boilerplate you want suppressed
EXTRA_STOPWORDS = {
    "quot", "amp", "www", "http", "https", "com", "re", "ve", "don",
    "just", "like", "know", "think", "really", "would", "could", "get",
    "one", "also", "even", "much", "still", "something", "anything",
}

# Palette
BG_COLOR   = "#0d1117"
HIST_COLOR = "#58a6ff"
WC_BG      = "#0d1117"

# ── LOAD DATA ─────────────────────────────────────────────────────────────────

print("Loading data …")
posts    = pd.read_csv(DATA_LOCATION + POSTS_CSV,    parse_dates=[POST_TS_COL])
comments = pd.read_csv(DATA_LOCATION + COMMENTS_CSV, parse_dates=[COMMENT_TS_COL])

print(f"  Posts   : {len(posts):,}")
print(f"  Comments: {len(comments):,}")

# ── 1. NORMALISED COMMENT VOLUME HISTOGRAM ────────────────────────────────────

print("\nBuilding normalised comment timestamps …")

# For each post, find the post timestamp and the time of the last comment
post_times     = posts.set_index(POST_ID_COL)[POST_TS_COL]
thread_end     = comments.groupby(COMMENT_POST_ID_COL)[COMMENT_TS_COL].max()

# Join onto comments
df = comments[[COMMENT_POST_ID_COL, COMMENT_TS_COL]].copy()
df = df.rename(columns={COMMENT_POST_ID_COL: "Post_ID", COMMENT_TS_COL: "comment_ts"})
df["post_ts"]   = df["Post_ID"].map(post_times)
df["thread_end"] = df["Post_ID"].map(thread_end)

# Drop rows where we cannot compute the normalised position
df = df.dropna(subset=["post_ts", "thread_end"])

# Duration in seconds
df["duration"] = (df["thread_end"] - df["post_ts"]).dt.total_seconds()

# Drop threads where post and last comment are at the same second (no span)
df = df[df["duration"] > 0]

# Normalised position: 0 = post time, 1 = last comment time
df["norm_pos"] = (df["comment_ts"] - df["post_ts"]).dt.total_seconds() / df["duration"]

# Clamp to [0, 1] to handle any minor clock drift / out-of-order rows
df["norm_pos"] = df["norm_pos"].clip(0, 1)

print(f"  Comments with valid normalised position: {len(df):,}")

# ── 2. BUILD STOPWORD SET ─────────────────────────────────────────────────────

base_sw = set(stopwords.words("english"))
all_sw  = base_sw | EXTRA_STOPWORDS | STOPWORDS  # STOPWORDS = wordcloud built-ins


def clean_text(series: pd.Series) -> str:
    """Concatenate a text series and strip URLs, punctuation, short tokens."""
    text = " ".join(series.dropna().astype(str))
    text = re.sub(r"http\S+|www\S+", " ", text)          # remove URLs
    text = re.sub(r"\[.*?\]|\(.*?\)", " ", text)          # remove [tags] (style)
    text = re.sub(r"[^a-zA-Z\s']", " ", text)             # keep letters + apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    # Remove single-char tokens
    text = " ".join(w for w in text.split() if len(w) > 2)
    return text


print("Cleaning post content …")
post_text    = clean_text(posts[POST_CONTENT_COL])

print("Cleaning comment content …")
comment_text = clean_text(comments[COMMENT_CONTENT_COL])

# ── 3. PLOT ───────────────────────────────────────────────────────────────────

print("\nRendering visualisations …")

plt.rcParams.update({
    "figure.facecolor":  BG_COLOR,
    "axes.facecolor":    BG_COLOR,
    "axes.edgecolor":    "#30363d",
    "axes.labelcolor":   "#e6edf3",
    "xtick.color":       "#8b949e",
    "ytick.color":       "#8b949e",
    "text.color":        "#e6edf3",
    "grid.color":        "#21262d",
    "font.family":       "monospace",
})

fig = plt.figure(figsize=(18, 16), facecolor=BG_COLOR)
gs  = gridspec.GridSpec(2, 2, figure=fig,
                        height_ratios=[1, 1],
                        hspace=0.4, wspace=0.3)

# ── Panel 1: Histogram (spans full top row) ───────────────────────────────────
ax_hist = fig.add_subplot(gs[0, :])

counts, edges, patches = ax_hist.hist(
    df["norm_pos"],
    bins=N_BINS,
    color=HIST_COLOR,
    edgecolor="#0d1117",
    linewidth=0.5,
    alpha=0.9,
)

# Colour-gradient on bars (early → late)
import numpy as np
norm_vals = (edges[:-1] + edges[1:]) / 2
cmap      = plt.cm.cool
for patch, val in zip(patches, norm_vals):
    patch.set_facecolor(cmap(val))

ax_hist.set_xlabel("Normalised Thread Time  (0 = post, 1 = final comment)", fontsize=12, labelpad=10)
ax_hist.set_ylabel("Number of Comments", fontsize=12, labelpad=10)
ax_hist.set_title("Comment Volume Over Normalised Thread Lifetime", fontsize=15, pad=16, fontweight="bold")
ax_hist.grid(axis="y", alpha=0.3)
ax_hist.set_xlim(0, 1)

# Annotate median
median_pos = df["norm_pos"].median()
ax_hist.axvline(median_pos, color="#f78166", linewidth=1.5, linestyle="--", alpha=0.8)
ax_hist.text(median_pos + 0.01, ax_hist.get_ylim()[1] * 0.92,
             f"median = {median_pos:.2f}",
             color="#f78166", fontsize=10)


In [ ]:
# ── 2. POST WORD CLOUD ────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("Generating post word cloud ...")

# Pass stopwords=set() because filtering is already done in clean_text.
# Passing the stopword set again here caused the double-filtering that
# emptied the previous version.
wc_posts = WordCloud(
    width=1600, height=900,
    background_color="#0d1117",
    stopwords=all_sw,
    colormap="cool",
    max_words=200,
    min_font_size=10,
    prefer_horizontal=0.8,
    collocations=True,
    relative_scaling=0.5,
).generate(post_text)

fig2, ax2 = plt.subplots(figsize=(16, 9), facecolor=BG_COLOR)
ax2.imshow(wc_posts, interpolation="bilinear")
ax2.axis("off")
ax2.set_title("Word Cloud — Posts", fontsize=16, pad=14,
              fontweight="bold", color="#e6edf3")
fig2.tight_layout(pad=0)
print("  Displaying post word cloud ...")
plt.show()

# ── 3. COMMENT WORD CLOUD ─────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("Generating comment word cloud ...")

wc_comments = WordCloud(
    width=1600, height=900,
    background_color="#0d1117",
    stopwords=all_sw,
    colormap="plasma",
    max_words=200,
    min_font_size=10,
    prefer_horizontal=0.8,
    collocations=True,
    relative_scaling=0.5,
).generate(comment_text)

fig3, ax3 = plt.subplots(figsize=(16, 9), facecolor=BG_COLOR)
ax3.imshow(wc_comments, interpolation="bilinear")
ax3.axis("off")
ax3.set_title("Word Cloud — Comments", fontsize=16, pad=14,
              fontweight="bold", color="#e6edf3")
fig3.tight_layout(pad=0)
print("  Displaying comment word cloud ...")
plt.show()

print("\n" + "=" * 60)
print("Done.")


In [ ]:
"""
Post → Sentence → Emotion Vectors
-----------------------------------
Grabs a single post by Post_ID, splits it into sentences, and runs each
sentence through borisn70/bert-43-multilabel-emotion-detection to produce
a 43-dimensional emotion vector per sentence.

Requirements:
    pip install pandas transformers torch nltk

The model is downloaded from HuggingFace on first run (~440 MB).
"""

import re
import pandas as pd
import numpy as np
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

nltk.download("punkt",        quiet=True)
nltk.download("punkt_tab",    quiet=True)
from nltk.tokenize import sent_tokenize

# ── CONFIG ────────────────────────────────────────────────────────────────────

POSTS_CSV       = "posts_cleaned.csv"
POST_ID_COL     = "Post_ID"
POST_CONTENT_COL = "Post_Content"
MODEL_NAME      = "borisn70/bert-43-multilabel-emotion-detection"

# Change this to any Post_ID you want to inspect.
# Set to None to just use the first post in the file.
TARGET_POST_ID  = None

# Sigmoid threshold — scores above this are considered "active" emotions
THRESHOLD = 0.5

# ── LOAD POST ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("Loading posts ...")
posts = pd.read_csv(DATA_LOCATION + POSTS_CSV)
print(f"  {len(posts):,} posts loaded")

if TARGET_POST_ID is not None:
    row = posts[posts[POST_ID_COL] == TARGET_POST_ID]
    if row.empty:
        raise ValueError(f"Post_ID '{TARGET_POST_ID}' not found in {POSTS_CSV}")
    post = row.iloc[0]
else:
    post = posts.iloc[0]

print(f"\n  Selected Post_ID : {post[POST_ID_COL]}")
print(f"  Content preview  : {str(post[POST_CONTENT_COL])[:120]} ...")

# ── SENTENCE SPLITTING ────────────────────────────────────────────────────────

raw_text = str(post[POST_CONTENT_COL])

# Light pre-clean before sentence splitting — remove quoted reply blocks and
# excessive whitespace, but keep the original words and punctuation so the
# sentence splitter has proper cues.
raw_text = re.sub(r"\[quote\].*?\[/quote\]", " ", raw_text,
                  flags=re.IGNORECASE | re.DOTALL)
raw_text = re.sub(r"<[^>]+>", " ", raw_text)
raw_text = re.sub(r"\[.*?\]", " ", raw_text)
raw_text = re.sub(r"\s+", " ", raw_text).strip()

sentences = sent_tokenize(raw_text)

# Drop very short fragments (punctuation-only, single words)
sentences = [s.strip() for s in sentences if len(s.split()) >= 3]

print(f"\n  Sentences found  : {len(sentences)}")
print("\n  Sentences:")
for i, s in enumerate(sentences):
    print(f"    [{i+1:02d}] {s}")

# ── LOAD MODEL ────────────────────────────────────────────────────────────────

print(f"\n{'=' * 60}")
print(f"Loading model: {MODEL_NAME}")
print("  (First run downloads ~440 MB — cached after that)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# Pull the label names in index order from the model config
id2label  = model.config.id2label                        # {0: 'admiration', ...}
n_labels  = len(id2label)
labels    = [id2label[i] for i in range(n_labels)]

print(f"  Labels ({n_labels}): {labels}")

# ── INFERENCE ─────────────────────────────────────────────────────────────────

print(f"\n{'=' * 60}")
print("Running inference ...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")
model.to(device)

emotion_vectors = []   # list of numpy arrays, one per sentence

with torch.no_grad():
    for i, sentence in enumerate(sentences):
        inputs  = tokenizer(sentence, return_tensors="pt",
                            truncation=True, max_length=512,
                            padding=True).to(device)
        outputs = model(**inputs)

        # The model uses a multilabel head — apply sigmoid (not softmax)
        probs   = torch.sigmoid(outputs.logits).squeeze().cpu().numpy()

        # Ensure always a 1-D array even for single-label edge cases
        probs   = np.atleast_1d(probs)
        emotion_vectors.append(probs)

        print(f"\n  Sentence {i+1:02d}: \"{sentence[:80]}{'...' if len(sentence)>80 else ''}\"")
        print(f"  {'Emotion':<22} {'Score':>6}   {'Bar'}")
        print(f"  {'-'*22} {'-'*6}   {'-'*20}")
        for label, score in sorted(zip(labels, probs), key=lambda x: -x[1]):
            bar    = "█" * int(score * 20)
            active = " ◄" if score >= THRESHOLD else ""
            print(f"  {label:<22} {score:>6.3f}   {bar}{active}")

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────

print(f"\n{'=' * 60}")
print("Emotion Vector Matrix (sentences × emotions)")
print(f"Shape: {len(emotion_vectors)} sentences × {n_labels} emotions\n")

matrix = np.vstack(emotion_vectors)   # (n_sentences, 43)
df_out = pd.DataFrame(matrix, columns=labels)
df_out.insert(0, "sentence", [f"S{i+1:02d}" for i in range(len(sentences))])

# Print full matrix
with pd.option_context("display.max_columns", None,
                        "display.width",       200,
                        "display.float_format", "{:.3f}".format):
    print(df_out.to_string(index=False))

# Mean vector across all sentences — the "post-level" emotion vector
print(f"\n{'=' * 60}")
print("Post-level emotion vector (mean across sentences):\n")
mean_vec = matrix.mean(axis=0)
for label, score in sorted(zip(labels, mean_vec), key=lambda x: -x[1]):
    bar    = "█" * int(score * 30)
    active = " ◄" if score >= THRESHOLD else ""
    print(f"  {label:<22} {score:.4f}  {bar}{active}")

print(f"\n{'=' * 60}")
print("Done.")
print(f"\nTo use the vectors downstream:")
print(f"  matrix   = np.array of shape ({len(sentences)}, {n_labels})  — per sentence")
print(f"  mean_vec = np.array of shape ({n_labels},)              — post level")